# 07 — Target Validation & Performance Normalization Reporting
---
**Purpose:** Analyzes the `Target_Tyre_Degradation` residual (calculated in Stage 05) to verify it behaves plausibly. Generates extensive diagnostic plots and validation reports across compounds, teams, circuits, and seasons.

**Input:** `outputs/features_engineered.parquet`
**Outputs:** 
- `outputs/plots/*.png`
- `outputs/target_validation_report.csv`
- `outputs/tyre_model_dataset.parquet`

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, warnings
warnings.filterwarnings("ignore")

# Set aesthetics for premium plots
plt.style.use('dark_background')
sns.set_palette("husl")

OUTPUT_DIR = os.path.join("..", "outputs")
PLOT_DIR = os.path.join(OUTPUT_DIR, "plots")
os.makedirs(PLOT_DIR, exist_ok=True)

input_path = os.path.join(OUTPUT_DIR, "data", "features_engineered.parquet")
print("Loading dataset...")
df = pd.read_parquet(input_path)

# Filter for clean modelling laps only to reduce noise in plots
plot_df = df[(df['is_clean_lap']) & (df['StintQualityFlag'] == 1) & (~df['flag_wet_compound'])].copy()
print(f"Loaded {len(plot_df):,} clean laps for validation.")

Loading dataset...
Loaded 79,745 clean laps for validation.


## 1. Diagnostic Plots

In [2]:
# 1. Target Distribution
plt.figure(figsize=(10, 6))
sns.histplot(plot_df['Target_Tyre_Degradation'], bins=100, kde=True, color='cyan')
plt.title('Target Distribution: Tyre Pace Residual')
plt.xlabel('Pace Deficit (Seconds)')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'target_distribution.png'))
plt.close()

# 2. Tyre Age vs Target
plt.figure(figsize=(12, 6))
sns.lineplot(data=plot_df, x='TyreAge', y='Target_Tyre_Degradation', ci='sd', color='magenta')
plt.title('Average Degradation Curve: Tyre Age vs Pace Deficit')
plt.xlabel('Physical Tyre Age (Laps)')
plt.ylabel('Pace Deficit (Seconds)')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'target_vs_tyreage.png'))
plt.close()

# 3. Target by Compound
plt.figure(figsize=(10, 6))
sns.boxplot(data=plot_df, x='Compound', y='Target_Tyre_Degradation', order=['SOFT', 'MEDIUM', 'HARD'])
plt.title('Pace Deficit by Compound')
plt.xlabel('Compound')
plt.ylabel('Pace Deficit (Seconds)')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'target_by_compound.png'))
plt.close()

# 4. Target by Team
plt.figure(figsize=(14, 6))
sns.boxplot(data=plot_df, x='CanonicalTeam', y='Target_Tyre_Degradation')
plt.title('Pace Deficit by Team (Tyre Management)')
plt.xticks(rotation=45)
plt.xlabel('Team')
plt.ylabel('Pace Deficit (Seconds)')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'target_by_team.png'))
plt.close()

# 5. Target by Circuit
plt.figure(figsize=(14, 6))
circuit_medians = plot_df.groupby('GrandPrix')['Target_Tyre_Degradation'].median().sort_values()
sns.boxplot(data=plot_df, x='GrandPrix', y='Target_Tyre_Degradation', order=circuit_medians.index)
plt.title('Pace Deficit by Circuit (Track Severity)')
plt.xticks(rotation=90)
plt.xlabel('Grand Prix')
plt.ylabel('Pace Deficit (Seconds)')
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, 'target_by_circuit.png'))
plt.close()

print("Diagnostic plots generated and saved to outputs/plots/")

Diagnostic plots generated and saved to outputs/plots/


## 2. Validation Reporting

In [3]:
# Generate aggregated metrics for the report
report_data = []

def analyze_target(dim_col):
    res = plot_df.groupby(dim_col)['Target_Tyre_Degradation'].agg(['count', 'mean', 'median', 'std']).reset_index()
    res['Dimension'] = dim_col
    res.rename(columns={dim_col: 'Value'}, inplace=True)
    return res

dimensions = ['Year', 'GrandPrix', 'Compound', 'CanonicalTeam']
for dim in dimensions:
    if dim in plot_df.columns:
        res = analyze_target(dim)
        report_data.append(res)

report_df = pd.concat(report_data, ignore_index=True)
report_df = report_df[['Dimension', 'Value', 'count', 'mean', 'median', 'std']]

out_report = os.path.join(OUTPUT_DIR, "reports", "target_validation_report.csv")
report_df.to_csv(out_report, index=False)

print(f"Validation report saved to {out_report}")
display(report_df.head(10))

Validation report saved to ..\outputs\target_validation_report.csv


,Dimension,Value,count,mean,median,std
0,Year,2022,17203,-0.388633,-0.6210,1.809266
1,Year,2023,20001,-0.021294,0.3640,3.110034
2,Year,2024,21244,1.111384,0.9465,1.613608
3,Year,2025,21297,0.865821,0.8710,1.831172
4,GrandPrix,Abu Dhabi Grand Prix,3976,0.585312,0.4580,1.430450
5,GrandPrix,Australian Grand Prix,2454,0.230952,0.1230,1.453509
6,GrandPrix,Austrian Grand Prix,4144,0.851115,0.9565,1.683051
7,GrandPrix,Azerbaijan Grand Prix,3338,1.229855,0.7355,2.061352
8,GrandPrix,Bahrain Grand Prix,3703,1.742637,1.6860,1.939560
9,GrandPrix,Belgian Grand Prix,2536,1.373785,1.3560,1.344867


## 3. Save Final Master Dataset

In [4]:
# Save the fully featured, engineered, and normalized master dataset
# (Using the requested filename `tyre_model_dataset.parquet`)
out_path = os.path.join(OUTPUT_DIR, "tyre_model_dataset.parquet")
df.to_parquet(out_path, index=False)
fsize = os.path.getsize(out_path) / 1e6
print(f"Saved master dataset to: {out_path} ({fsize:.1f} MB)")
print("\n[OK] Notebook 07 Target Validation complete.")

Saved master dataset to: ..\outputs\tyre_model_dataset.parquet (8.1 MB)

[OK] Notebook 07 Target Validation complete.
